In [ ]:
#@title Setup — Run this cell first
import warnings
warnings.filterwarnings('ignore')

# ── Embedded company_sim.py ──
"""
company_sim.py — Shared infrastructure for the Regression Autopsy course.
Embedded in every notebook's setup cell. Self-contained, no external deps
beyond numpy, pandas, matplotlib, statsmodels, ipywidgets.
"""

import json
import os
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence

import ipywidgets as widgets

# ---------------------------------------------------------------------------
# Color constants — consistent across all notebooks
# ---------------------------------------------------------------------------
COLORS = {
    'ols': '#2E5EA8',       # Blue — OLS estimator
    'truth': '#D4A017',     # Gold — true parameter
    'bias': '#C0392B',      # Red — bias/violation/error
    'repair': '#27AE60',    # Green — repair working
    'alt': '#7F8C8D',       # Gray — alternative estimators
}

# ---------------------------------------------------------------------------
# Gate System — module-level trap response storage
# ---------------------------------------------------------------------------
_trap_responses = {}
_TRAP_LOG_PATH = '/content/trap_log.json'


def _load_trap_log():
    """Load existing trap log from disk into memory."""
    global _trap_responses
    if os.path.exists(_TRAP_LOG_PATH):
        try:
            with open(_TRAP_LOG_PATH, 'r') as f:
                _trap_responses = json.load(f)
        except (json.JSONDecodeError, IOError):
            _trap_responses = {}


def _save_trap_log():
    """Persist in-memory trap responses to disk."""
    try:
        os.makedirs(os.path.dirname(_TRAP_LOG_PATH), exist_ok=True)
        with open(_TRAP_LOG_PATH, 'w') as f:
            json.dump(_trap_responses, f, indent=2)
    except IOError:
        pass  # Silently fail if /content not available (local dev)


def record_trap_response(notebook_id, question_id, response):
    """Save a trap response to the log."""
    key = f"{notebook_id}_{question_id}"
    _trap_responses[key] = {
        "response": response,
        "timestamp": datetime.now().isoformat(),
    }
    _save_trap_log()


def get_trap_response(notebook_id, question_id):
    """Retrieve a previously recorded trap response, or None."""
    key = f"{notebook_id}_{question_id}"
    entry = _trap_responses.get(key)
    return entry["response"] if entry else None


def check_gate(notebook_id, question_id):
    """Return True if a response has been recorded for this gate."""
    key = f"{notebook_id}_{question_id}"
    return key in _trap_responses


def get_all_responses():
    """Return all recorded trap responses."""
    return dict(_trap_responses)


def create_trap_widget(notebook_id, question_id, question_text, options):
    """Create a radio-button trap widget with submit button and gate logic."""
    label = widgets.HTML(f"<b>{question_text}</b>")
    radio = widgets.RadioButtons(
        options=options,
        value=None,
        layout=widgets.Layout(width='auto'),
    )
    submit = widgets.Button(description="Submit", button_style='primary')
    output = widgets.Output()

    def on_submit(btn):
        with output:
            output.clear_output()
            if radio.value is None:
                print("Please select an option before submitting.")
                return
            record_trap_response(notebook_id, question_id, radio.value)
            print(f"Response recorded: {radio.value}")
            submit.disabled = True
            radio.disabled = True

    submit.on_click(on_submit)

    # Pre-fill if already answered
    existing = get_trap_response(notebook_id, question_id)
    if existing is not None:
        try:
            radio.value = existing
            radio.disabled = True
            submit.disabled = True
        except Exception:
            pass

    return widgets.VBox([label, radio, submit, output])


# Load any existing responses at import time
_load_trap_log()


# ---------------------------------------------------------------------------
# CompanySimulator — NovaMart data generating process
# ---------------------------------------------------------------------------
class CompanySimulator:
    """
    Generates simulated NovaMart data with controllable pathologies:
    omitted variable bias, heteroscedasticity, nonlinearity, bad controls.
    """

    def __init__(
        self,
        endogeneity_strength=20,
        heteroscedasticity_strength=0.5,
        nonlinearity=True,
        bad_control_strength=0.1,
        noise_sigma=1.0,
    ):
        self.endogeneity_strength = endogeneity_strength
        self.heteroscedasticity_strength = heteroscedasticity_strength
        self.nonlinearity = nonlinearity
        self.bad_control_strength = bad_control_strength
        self.noise_sigma = noise_sigma

        # True DGP parameters
        self.beta_0 = 50
        self.beta_1 = 8      # coefficient on log(X1) or X1
        self.beta_2 = 3      # coefficient on X2
        self.beta_U = 2      # coefficient on U
        self.staff_loading = 5  # U -> X2 coefficient

    def generate(self, n=500, seed=42):
        """Generate observed data: revenue, ad_spend, staff_count, satisfaction."""
        rng = np.random.default_rng(seed)

        # Step 1: Generate U, noise terms, epsilon base
        U = rng.standard_normal(n)
        eta1 = rng.normal(0, self.noise_sigma, n)
        eta2 = rng.normal(0, self.noise_sigma, n)
        eta3 = rng.normal(0, self.noise_sigma, n)

        # Step 2: X1, X2 from U
        X1 = self.endogeneity_strength * U + eta1
        if self.nonlinearity:
            X1 = np.abs(X1) + 0.01  # Ensure positive for log
        X2 = self.staff_loading * U + eta2

        # Step 3: Compute Y
        # Heteroscedastic errors: eps ~ N(0, h * X1)
        eps_std = np.sqrt(np.abs(self.heteroscedasticity_strength * X1))
        eps = rng.normal(0, eps_std)

        if self.nonlinearity:
            Y = self.beta_0 + self.beta_1 * np.log(X1) + self.beta_2 * X2 + self.beta_U * U + eps
        else:
            Y = self.beta_0 + self.beta_1 * X1 + self.beta_2 * X2 + self.beta_U * U + eps

        # Step 4: X3 from Y (post-treatment / bad control)
        X3 = self.bad_control_strength * Y + eta3

        return pd.DataFrame({
            'revenue': Y,
            'ad_spend': X1,
            'staff_count': X2,
            'satisfaction': X3,
        })

    def truth(self, n=500, seed=42):
        """Generate data including hidden demand_U + return true parameter dict."""
        rng = np.random.default_rng(seed)

        U = rng.standard_normal(n)
        eta1 = rng.normal(0, self.noise_sigma, n)
        eta2 = rng.normal(0, self.noise_sigma, n)
        eta3 = rng.normal(0, self.noise_sigma, n)

        X1 = self.endogeneity_strength * U + eta1
        if self.nonlinearity:
            X1 = np.abs(X1) + 0.01
        X2 = self.staff_loading * U + eta2

        eps_std = np.sqrt(np.abs(self.heteroscedasticity_strength * X1))
        eps = rng.normal(0, eps_std)

        if self.nonlinearity:
            Y = self.beta_0 + self.beta_1 * np.log(X1) + self.beta_2 * X2 + self.beta_U * U + eps
        else:
            Y = self.beta_0 + self.beta_1 * X1 + self.beta_2 * X2 + self.beta_U * U + eps

        X3 = self.bad_control_strength * Y + eta3

        df = pd.DataFrame({
            'revenue': Y,
            'ad_spend': X1,
            'staff_count': X2,
            'satisfaction': X3,
            'demand_U': U,
        })

        params = {
            'beta_0': self.beta_0,
            'beta_1': self.beta_1,
            'beta_2': self.beta_2,
            'beta_U': self.beta_U,
            'sigma_epsilon': self.heteroscedasticity_strength,
        }

        return df, params

    def dgp_summary(self):
        """Return LaTeX specification string."""
        func = r"\log(X_1)" if self.nonlinearity else r"X_1"
        return (
            rf"$Y = {self.beta_0} + {self.beta_1} \cdot {func} "
            rf"+ {self.beta_2} \cdot X_2 + {self.beta_U} \cdot U + \varepsilon$"
            "\n\nWhere:\n"
            rf"- $U \sim N(0, 1)$ (unobserved market demand)"
            "\n"
            rf"- $X_1 = {self.endogeneity_strength} \cdot U + \eta_1$ (ad spend)"
            "\n"
            rf"- $X_2 = {self.staff_loading} \cdot U + \eta_2$ (staff count)"
            "\n"
            rf"- $X_3 = {self.bad_control_strength} \cdot Y + \eta_3$ (satisfaction, post-treatment)"
            "\n"
            rf"- $\varepsilon \sim N(0, {self.heteroscedasticity_strength} \cdot X_1)$"
            "\n"
            rf"- $\eta_i \sim N(0, {self.noise_sigma}^2)$"
        )

    def set_heteroscedasticity(self, strength):
        """Update heteroscedasticity strength."""
        self.heteroscedasticity_strength = strength

    def set_endogeneity(self, strength):
        """Update endogeneity strength (U -> X1 coefficient)."""
        self.endogeneity_strength = strength

    def set_nonlinearity(self, curvature):
        """Toggle or adjust nonlinearity. Pass bool or truthy value."""
        self.nonlinearity = bool(curvature)


# ---------------------------------------------------------------------------
# MonteCarloEngine — precompute simulation results for interactive sliders
# ---------------------------------------------------------------------------
class MonteCarloEngine:
    """Precomputes Monte Carlo draws across a parameter grid."""

    def run(self, estimator_fn, param_name, param_grid, simulator,
            n_reps=5000, n_obs=500):
        """
        For each value in param_grid, set param on simulator, run n_reps
        replications, collect estimator_fn output.

        Returns numpy array of shape (len(param_grid), n_reps).
        """
        results = np.empty((len(param_grid), n_reps))

        # Progress bar
        try:
            progress = widgets.IntProgress(
                value=0, min=0, max=len(param_grid),
                description='Simulating:', style={'description_width': 'initial'},
            )
            from IPython.display import display
            display(progress)
            use_widget = True
        except Exception:
            use_widget = False

        setter = getattr(simulator, f'set_{param_name}', None)

        for i, val in enumerate(param_grid):
            if setter is not None:
                setter(val)
            for r in range(n_reps):
                data = simulator.generate(n=n_obs, seed=r)
                results[i, r] = estimator_fn(data)
            if use_widget:
                progress.value = i + 1
            else:
                print(f"  [{i+1}/{len(param_grid)}] {param_name}={val:.3f}")

        if use_widget:
            progress.bar_style = 'success'

        return results

    def quick_run(self, estimator_fn, dgp_fn, n_reps=1000, n_obs=500):
        """
        Single-configuration simulation for sidebar mini-sims.
        dgp_fn(seed, n) -> DataFrame. No caching.
        """
        results = np.empty(n_reps)
        for r in range(n_reps):
            data = dgp_fn(seed=r, n=n_obs)
            results[r] = estimator_fn(data)
        return results


# ---------------------------------------------------------------------------
# DiagnosticSuite — standardized diagnostic visualizations
# ---------------------------------------------------------------------------
class DiagnosticSuite:
    """Produces four-panel residual diagnostics and summary test statistics."""

    @staticmethod
    def run_diagnostics(model_result):
        """
        2x2 diagnostic plot grid from a statsmodels OLS results object.
        Returns the matplotlib Figure.
        """
        fig, axes = plt.subplots(2, 2, figsize=(10, 8))

        influence = OLSInfluence(model_result)
        fitted = model_result.fittedvalues
        resid = model_result.resid
        std_resid = influence.resid_studentized_internal

        # Top-left: Residuals vs Fitted
        ax = axes[0, 0]
        ax.scatter(fitted, resid, alpha=0.4, s=15, color=COLORS['ols'])
        ax.axhline(0, color=COLORS['truth'], linewidth=1.5)
        ax.set_xlabel('Fitted values')
        ax.set_ylabel('Residuals')
        ax.set_title('Residuals vs Fitted')

        # Top-right: QQ plot
        ax = axes[0, 1]
        stats.probplot(resid, dist="norm", plot=ax)
        ax.set_title('Normal Q-Q')
        ax.get_lines()[0].set_color(COLORS['ols'])
        ax.get_lines()[1].set_color(COLORS['bias'])

        # Bottom-left: Scale-Location
        ax = axes[1, 0]
        sqrt_abs_std = np.sqrt(np.abs(std_resid))
        ax.scatter(fitted, sqrt_abs_std, alpha=0.4, s=15, color=COLORS['ols'])
        ax.set_xlabel('Fitted values')
        ax.set_ylabel(r'$\sqrt{|Standardized\ Residuals|}$')
        ax.set_title('Scale-Location')

        # Bottom-right: Residuals vs Leverage
        ax = axes[1, 1]
        leverage = influence.hat_matrix_diag
        cooks_d = influence.cooks_distance[0]
        ax.scatter(leverage, std_resid, alpha=0.4, s=15, color=COLORS['ols'])
        ax.axhline(0, color=COLORS['truth'], linewidth=1)
        ax.set_xlabel('Leverage')
        ax.set_ylabel('Standardized Residuals')
        ax.set_title("Residuals vs Leverage")

        # Cook's distance contours
        x_range = np.linspace(0.001, ax.get_xlim()[1], 100)
        for cook_val in [0.5, 1.0]:
            for sign in [1, -1]:
                p = model_result.df_model + 1
                y_val = sign * np.sqrt(cook_val * p * (1 - x_range) / x_range)
                ax.plot(x_range, y_val, '--', color=COLORS['bias'],
                        alpha=0.5, label=f"Cook's d={cook_val}" if sign == 1 else None)
        ax.legend(fontsize=8)

        fig.tight_layout()
        return fig

    @staticmethod
    def summary_tests(model_result):
        """
        Return dict of diagnostic test results:
        vif, breusch_pagan, durbin_watson, jarque_bera.
        """
        results = {}

        # VIF — requires exog with constant
        exog = model_result.model.exog
        exog_names = model_result.model.exog_names
        vif_dict = {}
        for i, name in enumerate(exog_names):
            if name == 'const':
                continue
            vif_dict[name] = variance_inflation_factor(exog, i)
        results['vif'] = vif_dict

        # Breusch-Pagan
        bp_stat, bp_pval, _, _ = het_breuschpagan(
            model_result.resid, model_result.model.exog
        )
        results['breusch_pagan'] = (bp_stat, bp_pval)

        # Durbin-Watson
        results['durbin_watson'] = durbin_watson(model_result.resid)

        # Jarque-Bera
        jb_stat, jb_pval, _, _ = jarque_bera(model_result.resid)
        results['jarque_bera'] = (jb_stat, jb_pval)

        return results


# ---------------------------------------------------------------------------
# AutopsyReport — widget factory for structured reflection
# ---------------------------------------------------------------------------
class AutopsyReport:
    """Creates reflection widgets for notebook wrap-up cells."""

    @staticmethod
    def lightweight(notebook_number):
        """Two-question mini autopsy for Notebooks 3-6."""
        threat = widgets.Text(
            description='Biggest threat:',
            placeholder='What is the biggest threat to this estimate?',
            layout=widgets.Layout(width='90%'),
            style={'description_width': '120px'},
        )
        check = widgets.Text(
            description='How to check:',
            placeholder='How would you check if that threat is real?',
            layout=widgets.Layout(width='90%'),
            style={'description_width': '120px'},
        )
        submit = widgets.Button(description='Save Autopsy', button_style='primary')
        output = widgets.Output()

        def on_submit(btn):
            with output:
                output.clear_output()
                nb_id = f"notebook_{notebook_number}"
                record_trap_response(nb_id, "threat", threat.value)
                record_trap_response(nb_id, "check", check.value)
                print("Autopsy responses saved.")
                submit.disabled = True

        submit.on_click(on_submit)
        return widgets.VBox([
            widgets.HTML(f"<h3>Mini Autopsy — Notebook {notebook_number}</h3>"),
            threat, check, submit, output
        ])

    @staticmethod
    def full(notebook_number, available_sidebars=None):
        """Full sensitivity-analysis autopsy for Notebooks 7-8."""
        fields = {
            'point_estimate': widgets.Text(
                description='Point estimate:',
                placeholder='My point estimate is:',
                layout=widgets.Layout(width='90%'),
                style={'description_width': '150px'},
            ),
            'robustness': widgets.Text(
                description='Robustness value:',
                placeholder='The robustness value is:',
                layout=widgets.Layout(width='90%'),
                style={'description_width': '150px'},
            ),
            'partial_r2': widgets.Text(
                description='Strongest partial R²:',
                placeholder='The strongest observed covariate has partial R² of:',
                layout=widgets.Layout(width='90%'),
                style={'description_width': '150px'},
            ),
            'plain_language': widgets.Text(
                description='Plain language:',
                placeholder='An omitted variable would need to be ___ times as important as ___ to explain away this result',
                layout=widgets.Layout(width='90%'),
                style={'description_width': '150px'},
            ),
        }

        children = [
            widgets.HTML(f"<h3>Full Autopsy Report — Notebook {notebook_number}</h3>"),
        ]
        children.extend(fields.values())

        if available_sidebars:
            sidebar_dropdown = widgets.Dropdown(
                options=['(select)'] + list(available_sidebars),
                description='Most analogous disaster:',
                layout=widgets.Layout(width='90%'),
                style={'description_width': '180px'},
            )
            children.append(sidebar_dropdown)
            fields['analogous_disaster'] = sidebar_dropdown

        submit = widgets.Button(description='Save Autopsy', button_style='primary')
        output = widgets.Output()

        def on_submit(btn):
            with output:
                output.clear_output()
                nb_id = f"notebook_{notebook_number}"
                for key, w in fields.items():
                    record_trap_response(nb_id, key, w.value)
                print("Full autopsy report saved.")
                submit.disabled = True

        submit.on_click(on_submit)
        children.extend([submit, output])
        return widgets.VBox(children)

# ── End embedded company_sim.py ──

from IPython.display import display, HTML, Markdown
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100
%matplotlib inline

# Fast mode: set FAST_TEST=1 to reduce precomputation
FAST_TEST = os.environ.get('FAST_TEST', '0') == '1'

# ── NB2-specific DGP ──
# True model: Y = 50 + 8*X1 + 3*X2 + eps
# Correct controls included. Heteroscedastic errors: Var(eps|X) = (h * X1)^2
NB2_BETA1_TRUE = 8.0

class NB2Simulator:
    """Linear DGP with controllable heteroscedasticity."""
    def __init__(self, hetero_strength=2.0):
        self.hetero_strength = hetero_strength

    def generate(self, n=500, seed=42):
        rng = np.random.default_rng(seed)
        X1 = rng.uniform(1, 20, n)  # ad_spend
        X2 = rng.standard_normal(n) * 2 + 5  # staff_count
        # Heteroscedastic errors: variance grows with X1
        eps_std = 1.0 + self.hetero_strength * (X1 / 20.0)
        eps = rng.normal(0, eps_std)
        Y = 50 + NB2_BETA1_TRUE * X1 + 3 * X2 + eps
        return pd.DataFrame({
            'revenue': Y, 'ad_spend': X1, 'staff_count': X2
        })

    def set_hetero(self, strength):
        self.hetero_strength = strength

nb2_sim = NB2Simulator(hetero_strength=2.0)

# ── Precompute MC cache for heteroscedasticity slider ──
print("Precomputing Monte Carlo simulations...")

def classical_coverage(data):
    """Run OLS, return 1 if classical 95% CI covers true beta1."""
    X = sm.add_constant(data[['ad_spend', 'staff_count']])
    model = sm.OLS(data['revenue'], X).fit()
    ci = model.conf_int(alpha=0.05).loc['ad_spend']
    return float(ci[0] <= NB2_BETA1_TRUE <= ci[1])

def robust_coverage(data):
    """Run OLS with HC3 robust SEs, return 1 if 95% CI covers true beta1."""
    X = sm.add_constant(data[['ad_spend', 'staff_count']])
    model = sm.OLS(data['revenue'], X).fit(cov_type='HC3')
    ci = model.conf_int(alpha=0.05).loc['ad_spend']
    return float(ci[0] <= NB2_BETA1_TRUE <= ci[1])

def get_classical_ci_width(data):
    X = sm.add_constant(data[['ad_spend', 'staff_count']])
    model = sm.OLS(data['revenue'], X).fit()
    ci = model.conf_int(alpha=0.05).loc['ad_spend']
    return ci[1] - ci[0]

def get_beta1(data):
    X = sm.add_constant(data[['ad_spend', 'staff_count']])
    return sm.OLS(data['revenue'], X).fit().params['ad_spend']

# Heteroscedasticity strength grid: 0 to 10, 20 positions
hetero_grid = np.round(np.linspace(0, 10, 21), 1)
N_REPS = 50 if FAST_TEST else 5000

mc_cache_classical = {}
mc_cache_robust = {}
mc_cache_beta = {}
progress = widgets.IntProgress(value=0, min=0, max=len(hetero_grid),
    description='Precomputing:', style={'description_width': 'initial'})
display(progress)

for i, h_val in enumerate(hetero_grid):
    sim_temp = NB2Simulator(hetero_strength=h_val)
    cov_c = np.empty(N_REPS)
    cov_r = np.empty(N_REPS)
    betas = np.empty(N_REPS)
    for r in range(N_REPS):
        data = sim_temp.generate(n=500, seed=r)
        cov_c[r] = classical_coverage(data)
        cov_r[r] = robust_coverage(data)
        betas[r] = get_beta1(data)
    mc_cache_classical[round(h_val, 1)] = cov_c
    mc_cache_robust[round(h_val, 1)] = cov_r
    mc_cache_beta[round(h_val, 1)] = betas
    progress.value = i + 1

progress.bar_style = 'success'

# Sample size cache for Limit cell
n_grid = [20, 30, 50, 75, 100, 150, 200, 300, 500]
mc_cache_n_classical = {}
mc_cache_n_robust = {}
progress2 = widgets.IntProgress(value=0, min=0, max=len(n_grid),
    description='Sample size:', style={'description_width': 'initial'})
display(progress2)

for i, n_val in enumerate(n_grid):
    sim_temp = NB2Simulator(hetero_strength=2.0)
    cov_c = np.empty(N_REPS)
    cov_r = np.empty(N_REPS)
    for r in range(N_REPS):
        data = sim_temp.generate(n=n_val, seed=r)
        cov_c[r] = classical_coverage(data)
        cov_r[r] = robust_coverage(data)
    mc_cache_n_classical[n_val] = cov_c
    mc_cache_n_robust[n_val] = cov_r
    progress2.value = i + 1

progress2.bar_style = 'success'
print("\nSetup complete!")


# Notebook 2: Why Your Uncertainty Is Wrong

---

## The NovaMart Scenario (continued)

You listened to Notebook 1. You included the right controls, avoided colliders, and got an unbiased coefficient for ad spend's effect on revenue. Your coefficient is $8.0 — every dollar of ad spend generates about $8 in revenue.

Now the CEO wants to know: *"How confident are we in that number?"* You look at the confidence interval from your regression and prepare to answer.

In [ ]:
# ── The Trap: Correct controls, but are the standard errors right? ──
data = nb2_sim.generate(n=500, seed=42)

X = sm.add_constant(data[['ad_spend', 'staff_count']])
model = sm.OLS(data['revenue'], X).fit()
print(model.summary())

ci = model.conf_int(alpha=0.05).loc['ad_spend']
print("\n" + "="*60)
print(f"  95% Confidence Interval for ad_spend: [{ci[0]:.1f}, {ci[1]:.1f}]")
print("="*60)

# ── Trap Widget ──
trap = create_trap_widget(
    notebook_id="notebook_2",
    question_id="q1",
    question_text="Are you confident the true effect is in this range?",
    options=[
        'A: Yes — 95% CI means 95% confident.',
        'B: No — the interval might be wrong.',
        'C: Depends on assumptions about the error structure.',
    ]
)
display(trap)


In [ ]:
# ── The Reveal ──
if not check_gate("notebook_2", "q1"):
    display(HTML('<div style="background:#FFCCCC;padding:15px;border-radius:8px;">'
                 '<b>Please answer the question in Cell 2 before continuing.</b></div>'))
else:
    response = get_trap_response("notebook_2", "q1")
    print(f"Your answer: {response}\n")

    if response.startswith("A"):
        display(HTML('<div style="background:#FFDDDD;padding:12px;border-radius:8px;">'
            '<b>Wrong!</b> The CI assumes homoscedastic errors. If that assumption fails, '
            'your "95%" interval might only cover 82% of the time.</div>'))
    elif response.startswith("B"):
        display(HTML('<div style="background:#FFF3CD;padding:12px;border-radius:8px;">'
            '<b>Right instinct, but why?</b> What specifically could make it wrong?</div>'))
    elif response.startswith("C"):
        display(HTML('<div style="background:#D4EDDA;padding:12px;border-radius:8px;">'
            '<b>Correct!</b> The classical CI requires homoscedastic errors. '
            "Let's see what happens when that assumption fails.</div>"))

    # ── Killer visualization: 1000 CIs stacked vertically ──
    sim = NB2Simulator(hetero_strength=2.0)
    n_ci = 1000
    fig, ax = plt.subplots(figsize=(10, 7))

    miss_count = 0
    # Plot a subset for visual clarity
    n_show = 200
    y_positions = np.arange(n_show)

    for idx in range(n_show):
        d = sim.generate(n=500, seed=idx)
        Xd = sm.add_constant(d[['ad_spend', 'staff_count']])
        m = sm.OLS(d['revenue'], Xd).fit()
        ci = m.conf_int(alpha=0.05).loc['ad_spend']
        covers = ci[0] <= NB2_BETA1_TRUE <= ci[1]
        color = COLORS['ols'] if covers else COLORS['bias']
        alpha = 0.3 if covers else 0.9
        ax.plot([ci[0], ci[1]], [idx, idx], color=color, alpha=alpha, linewidth=0.8)
        if not covers:
            miss_count += 1

    # Count full 1000
    total_miss = 0
    for idx in range(n_ci):
        d = sim.generate(n=500, seed=idx)
        Xd = sm.add_constant(d[['ad_spend', 'staff_count']])
        m = sm.OLS(d['revenue'], Xd).fit()
        ci = m.conf_int(alpha=0.05).loc['ad_spend']
        if not (ci[0] <= NB2_BETA1_TRUE <= ci[1]):
            total_miss += 1

    coverage = (n_ci - total_miss) / n_ci * 100
    ax.axvline(NB2_BETA1_TRUE, color=COLORS['truth'], linewidth=2.5,
               linestyle='-', label=f'True beta_1 = {NB2_BETA1_TRUE}')
    ax.set_xlabel('Estimated beta_1', fontsize=12)
    ax.set_ylabel('Simulation index', fontsize=12)
    ax.set_title(f'1,000 Classical 95% CIs — Actual Coverage: {coverage:.0f}%, not 95%!',
                 fontsize=13, fontweight='bold', color=COLORS['bias'])
    ax.legend(fontsize=10)

    # Coverage counter annotation
    ax.text(0.02, 0.98,
            f'Coverage: {coverage:.0f}%\nMisses: {total_miss}/{n_ci}',
            transform=ax.transAxes, fontsize=12, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', edgecolor=COLORS['bias'], alpha=0.9))

    fig.tight_layout()
    plt.show()

    # Formal statement
    display(HTML(
        '<div style="background:#F0F4FF;padding:20px;border-radius:10px;'
        'border-left:4px solid #2E5EA8;margin-top:15px;">'
        '<h3>The Standard Error Problem</h3>'
        '<p style="font-size:16px;">Classical:</p>'
        r'<p style="font-size:16px;">$$\text{Var}(\hat{\beta}) = \sigma^2 (X^\top X)^{-1}$$</p>'
        '<p style="font-size:16px;">Sandwich (robust):</p>'
        r'<p style="font-size:16px;">$$\text{Var}(\hat{\beta}) = (X^\top X)^{-1} X^\top \Omega X (X^\top X)^{-1}$$</p>'
        r'<p>When $\Omega \neq \sigma^2 I$ (heteroscedasticity), the classical formula is <b>wrong</b>. '
        'The coefficient is still unbiased, but your confidence intervals lie to you.</p>'
        '</div>'
    ))

# Trigger MathJax rendering for dynamically inserted LaTeX
try:
    from IPython.display import Javascript
    display(Javascript('if(window.MathJax){MathJax.Hub.Queue(["Typeset",MathJax.Hub]);}'))
except Exception:
    pass



In [ ]:
# ── Monte Carlo Confirmation ──
draws = mc_cache_beta[2.0]
cov_classical = mc_cache_classical[2.0].mean() * 100
cov_robust = mc_cache_robust[2.0].mean() * 100

fig_mc, ax_mc = plt.subplots(figsize=(9, 5))
bins = np.linspace(draws.min() - 0.5, draws.max() + 0.5, 60)

skip_btn = widgets.Button(description='Skip Animation', button_style='warning')
skip_flag = [False]
def on_skip(btn):
    skip_flag[0] = True
    btn.disabled = True
skip_btn.on_click(on_skip)
display(skip_btn)

n_frames = 200
batch = 25

for frame in range(n_frames):
    if skip_flag[0]:
        break
    n_show = min((frame + 1) * batch, len(draws))
    ax_mc.clear()
    ax_mc.hist(draws[:n_show], bins=bins, density=True,
               color=COLORS['ols'], alpha=0.7, edgecolor='white')
    ax_mc.axvline(NB2_BETA1_TRUE, color=COLORS['truth'], linewidth=2.5,
                  label=f'True beta_1 = {NB2_BETA1_TRUE}')
    ax_mc.set_xlabel('Estimated beta_hat_1', fontsize=11)
    ax_mc.set_ylabel('Density', fontsize=11)
    ax_mc.set_title(f'Monte Carlo ({n_show:,} / {len(draws):,} replications)', fontsize=12)
    ax_mc.legend(fontsize=9)
    fig_mc.canvas.draw()
    fig_mc.canvas.flush_events()

# Final
ax_mc.clear()
ax_mc.hist(draws, bins=bins, density=True,
           color=COLORS['ols'], alpha=0.7, edgecolor='white')
ax_mc.axvline(NB2_BETA1_TRUE, color=COLORS['truth'], linewidth=2.5,
              label=f'True beta_1 = {NB2_BETA1_TRUE}')
sim_mean = draws.mean()
ax_mc.axvline(sim_mean, color=COLORS['ols'], linewidth=2, linestyle='--',
              label=f'Sim mean = {sim_mean:.2f}')
ax_mc.set_xlabel('Estimated beta_hat_1', fontsize=11)
ax_mc.set_ylabel('Density', fontsize=11)
ax_mc.set_title(f'Coefficient is Unbiased — But CIs Are Wrong!', fontsize=12, fontweight='bold')
ax_mc.legend(fontsize=9)
fig_mc.tight_layout()
plt.show()

print(f"\n  Coefficient: E[beta_hat_1] = {sim_mean:.2f} (true = {NB2_BETA1_TRUE})")
print(f"  Classical 95% CI coverage: {cov_classical:.1f}% (should be 95%)")
print(f"  Robust 95% CI coverage:    {cov_robust:.1f}% (should be 95%)")


---
## Discussion Prompt

> **Your manager says the confidence interval is narrow so we're confident. When is this misleading?**

Think about:
- A narrow CI that's *wrong* vs. a wide CI that's *correct*
- What does "confidence" actually mean in frequentist statistics?
- Can you have a narrow CI with the wrong coverage?

In [ ]:
# ── The Destruction Playground: Heteroscedasticity strength slider ──
fig_dest, (ax_cov, ax_summ) = plt.subplots(1, 2, figsize=(13, 5),
    gridspec_kw={'width_ratios': [2, 1]})
plt.subplots_adjust(wspace=0.35)

hetero_slider = widgets.SelectionSlider(
    options=[round(v, 1) for v in np.linspace(0, 10, 21)],
    value=2.0,
    description='Hetero strength:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='500px'),
)

def update_destruction(change):
    h_val = change['new'] if isinstance(change, dict) else change
    h_key = round(h_val, 1)
    cov_c = mc_cache_classical[h_key].mean() * 100
    cov_r = mc_cache_robust[h_key].mean() * 100

    # Left: coverage vs hetero strength
    ax_cov.clear()
    h_keys = sorted(mc_cache_classical.keys())
    covs_c = [mc_cache_classical[k].mean() * 100 for k in h_keys]
    covs_r = [mc_cache_robust[k].mean() * 100 for k in h_keys]

    ax_cov.plot(h_keys, covs_c, color=COLORS['ols'], linewidth=2, marker='o',
                markersize=4, label='Classical SE')
    ax_cov.plot(h_keys, covs_r, color=COLORS['repair'], linewidth=2, marker='s',
                markersize=4, linestyle='-.', label='Robust (HC3) SE')
    ax_cov.axhline(95, color=COLORS['truth'], linewidth=2, linestyle='--',
                   label='Nominal 95%')
    ax_cov.axvline(h_val, color=COLORS['bias'], linewidth=1.5, linestyle=':', alpha=0.7)
    ax_cov.set_xlabel('Heteroscedasticity Strength', fontsize=11)
    ax_cov.set_ylabel('Coverage (%)', fontsize=11)
    ax_cov.set_title(f'CI Coverage (hetero = {h_val:.1f})',
                     fontsize=12, fontweight='bold')
    ax_cov.set_ylim(70, 100)
    ax_cov.legend(fontsize=9)

    # Right: summary
    ax_summ.clear()
    ax_summ.axis('off')
    gap = 95 - cov_c
    status = "WARNING" if gap > 3 else "OK"
    color = COLORS['bias'] if gap > 3 else COLORS['repair']
    summary_text = (
        f"Hetero strength = {h_val:.1f}\n\n"
        f"Classical coverage: {cov_c:.1f}%\n"
        f"Robust coverage:    {cov_r:.1f}%\n"
        f"Target:             95.0%\n\n"
        f"Classical gap: {gap:+.1f}pp\n\n"
        f"{status}"
    )
    ax_summ.text(0.1, 0.5, summary_text, fontsize=13,
                 verticalalignment='center', fontfamily='monospace',
                 bbox=dict(boxstyle='round,pad=0.8', facecolor='#F8F8F8',
                           edgecolor=color, linewidth=2))
    ax_summ.set_title('Summary', fontsize=12, fontweight='bold')

    fig_dest.canvas.draw_idle()

hetero_slider.observe(update_destruction, names='value')
update_destruction({'new': 2.0})
display(hetero_slider)
plt.show()


In [ ]:
# ── The Repair: Classical vs Robust SEs ──
fig_rep, (ax_class, ax_rob) = plt.subplots(1, 2, figsize=(13, 5))
plt.subplots_adjust(wspace=0.35)

toggle_robust = widgets.ToggleButton(value=False,
    description='Use Robust (HC3) Standard Errors',
    button_style='', layout=widgets.Layout(width='300px'))

hetero_repair = widgets.SelectionSlider(
    options=[round(v, 1) for v in np.linspace(0, 10, 21)],
    value=2.0, description='Hetero:', style={'description_width': '50px'},
    layout=widgets.Layout(width='400px'))

def update_repair(change=None):
    h_val = round(hetero_repair.value, 1)
    betas = mc_cache_beta[h_val]
    cov_c = mc_cache_classical[h_val].mean() * 100
    cov_r = mc_cache_robust[h_val].mean() * 100
    bins = np.linspace(betas.min() - 0.3, betas.max() + 0.3, 50)

    ax_class.clear()
    ax_class.hist(betas, bins=bins, density=True,
                  color=COLORS['ols'], alpha=0.7, edgecolor='white')
    ax_class.axvline(NB2_BETA1_TRUE, color=COLORS['truth'], linewidth=2.5)
    ax_class.set_title(f'Classical SEs\nCoverage: {cov_c:.1f}%',
                       fontsize=11, color=COLORS['bias'] if cov_c < 92 else 'black')
    ax_class.set_xlabel('beta_hat_1')
    ax_class.set_ylabel('Density')

    if toggle_robust.value:
        ax_rob.clear()
        ax_rob.hist(betas, bins=bins, density=True,
                    color=COLORS['repair'], alpha=0.7, edgecolor='white')
        ax_rob.axvline(NB2_BETA1_TRUE, color=COLORS['truth'], linewidth=2.5)
        ax_rob.set_title(f'Robust (HC3) SEs\nCoverage: {cov_r:.1f}%',
                         fontsize=11, color=COLORS['repair'])
        toggle_robust.button_style = 'success'
    else:
        ax_rob.clear()
        ax_rob.text(0.5, 0.5, 'Toggle Robust SEs\nto activate',
                    ha='center', va='center', fontsize=12, color='gray',
                    transform=ax_rob.transAxes)
        ax_rob.set_title('Robust (HC3) SEs', fontsize=11, color='gray')
        toggle_robust.button_style = ''
    ax_rob.set_xlabel('beta_hat_1')
    ax_rob.set_ylabel('Density')

    fig_rep.canvas.draw_idle()

toggle_robust.observe(update_repair, names='value')
hetero_repair.observe(update_repair, names='value')
update_repair()
display(toggle_robust)
display(hetero_repair)
plt.show()


In [ ]:
# ── The Limit: Sample size slider ──
fig_lim, (ax_lcover, ax_lsumm) = plt.subplots(1, 2, figsize=(13, 5),
    gridspec_kw={'width_ratios': [2, 1]})
plt.subplots_adjust(wspace=0.35)

n_slider = widgets.SelectionSlider(
    options=[20, 30, 50, 75, 100, 150, 200, 300, 500],
    value=100,
    description='Sample size n:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='500px'),
)

def update_limit(change=None):
    n_val = n_slider.value

    # Coverage vs sample size
    ax_lcover.clear()
    n_keys = sorted(mc_cache_n_classical.keys())
    covs_c = [mc_cache_n_classical[k].mean() * 100 for k in n_keys]
    covs_r = [mc_cache_n_robust[k].mean() * 100 for k in n_keys]

    ax_lcover.plot(n_keys, covs_c, color=COLORS['ols'], linewidth=2, marker='o',
                   markersize=5, label='Classical SE')
    ax_lcover.plot(n_keys, covs_r, color=COLORS['repair'], linewidth=2, marker='s',
                   markersize=5, linestyle='-.', label='Robust (HC3) SE')
    ax_lcover.axhline(95, color=COLORS['truth'], linewidth=2, linestyle='--',
                      label='Nominal 95%')
    ax_lcover.axvline(n_val, color=COLORS['bias'], linewidth=1.5, linestyle=':', alpha=0.7)
    ax_lcover.set_xlabel('Sample Size (n)', fontsize=11)
    ax_lcover.set_ylabel('Coverage (%)', fontsize=11)
    ax_lcover.set_xscale('log')
    ax_lcover.set_ylim(70, 100)
    ax_lcover.legend(fontsize=9)

    cov_c = mc_cache_n_classical[n_val].mean() * 100
    cov_r = mc_cache_n_robust[n_val].mean() * 100
    robust_better = cov_r > cov_c or abs(cov_r - 95) < abs(cov_c - 95)

    title_status = "Repair effective" if robust_better and n_val >= 50 else "Repair unreliable"
    title_color = COLORS['repair'] if "effective" in title_status else COLORS['bias']
    ax_lcover.set_title(f'{title_status} (n = {n_val})',
                        fontsize=12, fontweight='bold', color=title_color)

    ax_lsumm.clear()
    ax_lsumm.axis('off')
    summary_text = (
        f"n = {n_val}\n\n"
        f"Classical: {cov_c:.1f}%\n"
        f"Robust:    {cov_r:.1f}%\n"
        f"Target:    95.0%\n\n"
        f"{'Robust wins' if robust_better else 'Classical wins!'}\n"
        f"{'(small n: robust unreliable)' if n_val < 50 else ''}"
    )
    ax_lsumm.text(0.1, 0.5, summary_text, fontsize=13,
                  verticalalignment='center', fontfamily='monospace',
                  bbox=dict(boxstyle='round,pad=0.8', facecolor='#F8F8F8',
                            edgecolor=title_color, linewidth=2))
    ax_lsumm.set_title('Summary', fontsize=12, fontweight='bold')

    fig_lim.canvas.draw_idle()

n_slider.observe(update_limit, names='value')
update_limit()
display(n_slider)
plt.show()


In [ ]:
#@title Real-World Disaster: Google Flu Trends

story_html = (
    '<div style="padding:15px;font-size:14px;line-height:1.6;">'
    '<h4>Google Flu Trends: When Autocorrelation Killed Uncertainty</h4>'
    '<p>Google Flu Trends used search query data to predict flu prevalence. '
    'Their confidence intervals were incredibly narrow — suggesting remarkable precision. '
    'But the data was <b>autocorrelated</b>: this week\'s searches were similar to last week\'s.</p>'
    '<p>The effective sample size was a fraction of the actual sample size. '
    'Google Flu Trends dramatically overestimated flu prevalence in 2012-2013, '
    'missing by more than 2x, despite "tight" confidence intervals.</p>'
    '</div>'
)

math_html = (
    '<div style="padding:15px;font-size:14px;line-height:1.8;">'
    '<h4>Effective Sample Size Under Autocorrelation</h4>'
    r'<p>$$n_{\text{eff}} = n \cdot \frac{1-\rho}{1+\rho}$$</p>'
    r'<p>With $\rho = 0.8$ (autocorrelation) and $n = 100$:</p>'
    r'<p>$$n_{\text{eff}} = 100 \cdot \frac{1-0.8}{1+0.8} = 100 \cdot \frac{0.2}{1.8} \approx 11$$</p>'
    '<p><b>100 observations behave like 11!</b> Your standard errors are '
    r'$\sqrt{100/11} \approx 3\times$ too small.</p>'
    '</div>'
)

sim_output = widgets.Output()
sim_btn = widgets.Button(description='Run Autocorrelation Sim', button_style='primary')

def run_gft_sim(btn):
    with sim_output:
        sim_output.clear_output(wait=True)
        engine = MonteCarloEngine()

        def ar1_dgp(seed, n):
            rng = np.random.default_rng(seed)
            rho_ar = 0.8
            X = np.zeros(n)
            for t in range(1, n):
                X[t] = rho_ar * X[t-1] + rng.standard_normal()
            Y = 2.0 * X + rng.standard_normal(n)
            return pd.DataFrame({'Y': Y, 'X': X})

        def ar1_coverage(data):
            X = sm.add_constant(data['X'])
            m = sm.OLS(data['Y'], X).fit()
            ci = m.conf_int(alpha=0.05).loc['X']
            return float(ci[0] <= 2.0 <= ci[1])

        results = engine.quick_run(ar1_coverage, ar1_dgp, n_reps=(50 if FAST_TEST else 1000), n_obs=100)

        coverage = results.mean() * 100
        fig, ax = plt.subplots(figsize=(6, 3))
        ax.bar(['Classical CI\n(assumes independence)', 'Nominal'],
               [coverage, 95],
               color=[COLORS['bias'], COLORS['truth']], alpha=0.8)
        ax.set_ylabel('Coverage (%)')
        ax.set_title(f'Autocorrelation Destroys Coverage: {coverage:.0f}% vs 95%',
                     fontweight='bold')
        ax.set_ylim(0, 100)
        fig.tight_layout()
        plt.show()
        print(f"  Actual coverage: {coverage:.1f}% (should be 95%)")
        print(f"  Effective n: ~{100 * 0.2/1.8:.0f} out of 100")

sim_btn.on_click(run_gft_sim)

tab = widgets.Tab(children=[
    widgets.HTML(story_html),
    widgets.HTML(math_html),
    widgets.VBox([sim_btn, sim_output]),
])
tab.set_title(0, 'Story')
tab.set_title(1, 'Math')
tab.set_title(2, 'Simulation')
display(tab)


In [ ]:
# ── Mini Autopsy ──
autopsy = AutopsyReport.lightweight(2)
display(autopsy)

---
## What's Next?

Your standard errors are correct now — you're using robust standard errors that account for heteroscedasticity. **But what does "significant" actually mean?**

With enough data, *everything* is significant. The question isn't whether an effect exists — it's whether it *matters*.

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 12px; color: white; margin-top: 20px;">
<h3 style="color: white; margin-top: 0;">Notebook 3: Why Your Significance Is Wrong</h3>
<p>Your uncertainty is fixed. Now let's break your interpretation of significance.</p>
<p><a href="03_significance.ipynb" style="color: #FFD700; font-weight: bold;">Open Notebook 3</a></p>
</div>